# Logistic regression for subsequent-shot prediction

This notebook fits a small current-state baseline: given an observed action,
estimate whether the same team will take a subsequent recorded shot before
the possession ends. A goal is not required. Only current action type, position
and timing enter the model.

Run the cells in order after installing the modelling extra and running
preprocessing. Saved outputs provide a reference. The [baseline documentation](../docs/logistic-baseline.md)
defines the labels, weighting and limitations. This is chronological development
validation within 2023, not the final evaluation on later seasons.

## 1. Environment and modelling functions

The setup locates the repository and imports the labelling and modelling functions.
The notebook uses the same implementation as the command-line baseline.
Install `.[model]` in the project environment before running this cell.

In [1]:
from pathlib import Path
import pandas as pd
import sklearn
from IPython.display import display

from afl_opposition.baseline import load_preprocessed, train_baseline, save_baseline
from afl_opposition.examples import build_examples, chronological_split
from afl_opposition.preprocess import FEATURE_COLUMNS

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "data/sources.json").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
print("scikit-learn:", sklearn.__version__)
print("Model inputs:", FEATURE_COLUMNS)

scikit-learn: 1.9.1
Model inputs: ['action_type', 'x_norm', 'y_norm', 'gap_seconds', 'elapsed_seconds']


## 2. Processed-data verification

The loader verifies the raw snapshot checksum and preprocessing provenance,
checks table counts and audits retrospective shot markers against earlier
flagged kicks. A conflicting shot marker stops the run for label review.
These audits do not create additional model inputs.

In [2]:
SEASON = 2023
events, possessions, provenance = load_preprocessed(project_root, SEASON)
print("Processed actions:", len(events))
print("Inferred possessions:", len(possessions))
print("Verified retrospective shot markers:", provenance["verified_retrospective_shot_markers"])

Processed actions: 317812
Inferred possessions: 56160
Verified retrospective shot markers: 2000


## 3. Labels and eligibility rules

For a possession whose first shot is event 4, the first three observed states
receive target 1. The shot event and all later states are excluded. Completed
episodes with no recorded shot receive target 0. Reviewed or censored episodes
are excluded; a goal or behind without a shot flag is also excluded.

Every example references one current action. Its prefix remains identifiable
through `possession_id` and `event_index`, but this baseline does not consume
the preceding action sequence. Each episode's weights sum to 1.

In [3]:
dataset = build_examples(events, possessions)
examples = dataset.examples
display(pd.Series(dataset.summary["status_counts"], name="possessions").to_frame())
display(pd.Series({
    "Included possessions": dataset.summary["included_possessions"],
    "Positive possessions": dataset.summary["positive_possessions"],
    "Negative possessions": dataset.summary["negative_possessions"],
    "Current-state examples": len(examples),
}, name="count").to_frame())
print("Maximum weight-sum error:", float(
    (examples.groupby("possession_id")["sample_weight"].sum() - 1).abs().max()
))

,possessions
eligible,52429
preprocessing_review,3698
no_pre_shot_action,32
score_without_shot_flag,1


,count
Included possessions,52429
Positive possessions,10267
Negative possessions,42162
Current-state examples,285771


Maximum weight-sum error: 0.0


## 4. Positive and negative examples

The first table shows the raw processed actions in a selected positive episode,
including the shot marker for inspection. The second contains only the eligible
pre-shot states and their labels. The negative example has a supported endpoint
without a recorded shot. Shot flags are absent from the model feature list.

In [4]:
positive_id = examples.loc[examples["target"].eq(1), "possession_id"].iloc[0]
negative_id = examples.loc[examples["target"].eq(0), "possession_id"].iloc[0]
print("Positive possession: processed actions")
display(events.loc[events["possession_id"].eq(positive_id), [
    "event_index", "description", "period_seconds", "is_explicit_shot",
]])
print("Positive possession: eligible model examples")
display(examples.loc[examples["possession_id"].eq(positive_id), [
    "event_index", *FEATURE_COLUMNS, "target", "sample_weight",
]])
print("Negative possession: eligible model examples")
display(examples.loc[examples["possession_id"].eq(negative_id), [
    "event_index", *FEATURE_COLUMNS, "target", "sample_weight",
]])

Positive possession: processed actions


,event_index,description,period_seconds,is_explicit_shot
2,0,Free For,24,False
3,1,Kick,26,False
4,2,Loose Ball Get Crumb,32,False
5,3,Kick,32,True


Positive possession: eligible model examples


,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,target,sample_weight
2,0,free_kick,-0.0375,-0.028369,0,0,1,0.333333
3,1,kick,-0.1,-0.099291,2,2,1,0.333333
4,2,loose_ball_get,0.575,-0.070922,6,8,1,0.333333


Negative possession: eligible model examples


,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,target,sample_weight
0,0,hard_ball_get,-0.025,0.014184,0,0,0,0.5
1,1,handball,-0.025,0.0,0,0,0,0.5


## 5. Chronological match split

The cutoff approximates 70% of the matches for training. All matches on the
cutoff date enter validation, so dates and matches do not overlap between
partitions. The later matches measure development performance; there is no
hyperparameter search or final test evaluation in this notebook.

In [5]:
TRAIN_FRACTION = 0.7
match_split = chronological_split(events, train_fraction=TRAIN_FRACTION)
display(match_split.groupby("split").agg(
    matches=("match_id", "size"), first_date=("date", "min"), last_date=("date", "max"),
))
assert match_split["match_id"].is_unique
assert match_split.groupby("date")["split"].nunique().eq(1).all()
assert (
    match_split.loc[match_split["split"].eq("train"), "date"].max()
    < match_split.loc[match_split["split"].eq("validation"), "date"].min()
)

,matches,first_date,last_date
split,,,
train,151,2023-03-16,2023-07-15
validation,65,2023-07-16,2023-09-30


## 6. Current-state model fitting

The pipeline one-hot encodes the current action and standardises the four
numeric columns. Both transformations learn from training rows only. An
unseen validation action produces zeros in the action block.

Logistic regression uses L2 regularisation with C=1 and the lbfgs solver.
Prefix weights give each eligible possession equal influence. Numerical
scaling uses unweighted training rows. No lagged action columns are created.
The example below displays the fitted dummy variables and scaled values.

In [6]:
result = train_baseline(events, possessions, train_fraction=TRAIN_FRACTION)
pipeline = result.pipeline
transformer = pipeline.named_steps["features"]
feature_names = transformer.get_feature_names_out()
print("Encoded feature count:", len(feature_names))
print("Optimisation iterations:", result.summary["iterations"])
preview = result.examples.head(6)
encoded = pd.DataFrame(transformer.transform(preview[FEATURE_COLUMNS]), columns=feature_names)
display(encoded)
display(result.coefficients)
print("Intercept:", result.summary["intercept"])

Encoded feature count: 16
Optimisation iterations: 29


,action__action_type_bounce,action__action_type_free_kick,action__action_type_gather,action__action_type_ground_kick,action__action_type_handball,action__action_type_hard_ball_get,action__action_type_kick,action__action_type_kick_in,action__action_type_kick_in_play_on,action__action_type_loose_ball_get,action__action_type_mark,action__action_type_receive,numeric__x_norm,numeric__y_norm,numeric__gap_seconds,numeric__elapsed_seconds
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.186191,0.024367,-0.633184,-0.775996
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.186191,-0.003410,-0.633184,-0.775996
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.158978,-0.058964,-0.633184,-0.775996
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.022912,-0.197849,0.042217,-0.580384
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.492416,-0.142295,1.393020,0.006450
5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.213404,0.107698,-0.633184,-0.775996


,feature,coefficient
0,action__action_type_bounce,0.937254
1,action__action_type_free_kick,0.490225
2,action__action_type_gather,-0.247062
3,action__action_type_ground_kick,-1.722936
4,action__action_type_handball,-0.420535
5,action__action_type_hard_ball_get,-0.768644
6,action__action_type_kick,-0.317656
7,action__action_type_kick_in,0.026806
8,action__action_type_kick_in_play_on,0.079888
9,action__action_type_loose_ball_get,-0.205434


Intercept: -1.615862960689281


## 7. Constant-baseline comparison

The constant model predicts the weighted training shot rate for every example.
The fitted classifier is evaluated using the same possession weights. Higher
average precision and lower log loss and Brier score indicate improvement.
Average precision is a precision-recall summary, not trapezoidal PR-AUC.

Training scores describe fit; validation scores describe performance on later
2023 matches. They do not establish performance on future seasons or during
live matches. Prefixes from one possession are correlated, so they must not
be treated as independent observations for confidence intervals.

In [7]:
print("Constant training probability:", f"{result.summary['constant_probability']:.4f}")
display(result.metrics.round(4))
display(result.coverage.pivot_table(
    index=["split", "team_id"], columns="status", values="possessions", fill_value=0,
))

Constant training probability: 0.1962


,split,model,examples,possessions,observed_shot_rate,average_precision,log_loss,brier_score
0,train,constant,199809,36794,0.1962,0.1962,0.4951,0.1577
1,train,logistic,199809,36794,0.1962,0.5439,0.3948,0.1237
2,validation,constant,85962,15635,0.1949,0.1949,0.4932,0.1569
3,validation,logistic,85962,15635,0.1949,0.5282,0.4002,0.1253


status               eligible  no_pre_shot_action  preprocessing_review  \
split      team_id                                                        
train      CD_T10      2152.0                 2.0                 155.0   
           CD_T100     1865.0                 0.0                 150.0   
           CD_T1000    2082.0                 1.0                 146.0   
           CD_T1010    2160.0                 1.0                 158.0   
           CD_T110     2059.0                 1.0                 157.0   
           CD_T120     2034.0                 2.0                 159.0   
           CD_T130     2088.0                 2.0                 181.0   
           CD_T140     2088.0                 1.0                 139.0   
           CD_T150     1778.0                 1.0                 126.0   
           CD_T160     2040.0                 0.0                 156.0   
           CD_T20      2102.0                 1.0                 139.0   
           CD_T30      2106.0                 0.0                 126.0   
           CD_T40      2038.0                 1.0                 157.0   
           CD_T50      2006.0                 0.0                 126.0   
           CD_T60      2039.0                 0.0                 135.0   
           CD_T70      2083.0                 4.0                 139.0   
           CD_T80      1879.0                 1.0                 141.0   
           CD_T90      2195.0                 1.0                 140.0   
validation CD_T10       760.0                 3.0                  43.0   
           CD_T100      798.0                 2.0                  59.0   
           CD_T1000     726.0                 1.0                  40.0   
           CD_T1010    1157.0                 1.0                  88.0   
           CD_T110      944.0                 1.0                  52.0   
           CD_T120      813.0                 0.0                  61.0   
           CD_T130      825.0                 0.0                  59.0   
           CD_T140      740.0                 0.0                  50.0   
           CD_T150      812.0                 0.0                  48.0   
           CD_T160      869.0                 2.0                  59.0   
           CD_T20      1090.0                 1.0                  63.0   
           CD_T30      1159.0                 0.0                  95.0   
           CD_T40      1095.0                 0.0                  80.0   
           CD_T50       679.0                 0.0                  39.0   
           CD_T60       661.0                 0.0                  48.0   
           CD_T70       687.0                 0.0                  46.0   
           CD_T80       803.0                 2.0                  59.0   
           CD_T90      1017.0                 0.0                  79.0   

status               score_without_shot_flag  
split      team_id                            
train      CD_T10                        0.0  
           CD_T100                       0.0  
           CD_T1000                      0.0  
           CD_T1010                      0.0  
           CD_T110                       0.0  
           CD_T120                       0.0  
           CD_T130                       0.0  
           CD_T140                       0.0  
           CD_T150                       0.0  
           CD_T160                       0.0  
           CD_T20                        0.0  
           CD_T30                        1.0  
           CD_T40                        0.0  
           CD_T50                        0.0  
           CD_T60                        0.0  
           CD_T70                        0.0  
           CD_T80                        0.0  
           CD_T90                        0.0  
validation CD_T10                        0.0  
           CD_T100                       0.0  
           CD_T1000                      0.0  
           CD_T1010                      0.0  
           CD_T110   

## 8. Validation calibration

Fixed probability bins summarise validation calibration. Each bin compares
the weighted mean prediction with the weighted observed shot rate.
Close agreement is desirable. These are diagnostics; no recalibration is fitted.
Bins with little total weight provide weaker evidence.

In [8]:
display(result.calibration.loc[
    result.calibration["split"].eq("validation") & result.calibration["model"].eq("logistic"),
    ["bin_lower", "bin_upper", "examples", "weight_sum", "mean_prediction", "observed_shot_rate"],
].round(4))

,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_shot_rate
12,0.0,0.1,32115,6308.7958,0.0469,0.0553
13,0.1,0.2,21116,3715.7941,0.1458,0.1525
14,0.2,0.3,11494,1933.5015,0.2461,0.2335
15,0.3,0.4,7577,1316.1111,0.3475,0.3098
16,0.4,0.5,5519,986.3286,0.4471,0.3858
17,0.5,0.6,3729,697.8852,0.5456,0.5302
18,0.6,0.7,2483,403.9893,0.6440,0.6879
19,0.7,0.8,1432,226.4115,0.7417,0.8861
20,0.8,0.9,474,44.8727,0.8307,0.9366
21,0.9,1.0,23,1.3103,0.9165,0.8766


## 9. Predictions through held-out possessions

The examples are the first chronological validation possession of each target
class with at least three eligible states. The selection is deterministic and does
not depend on prediction quality. Probabilities can rise or fall as the current
action, location and timing change. The model does not see previous action
identities or the eventual target when producing these probabilities.

In [9]:
validation = result.predictions.loc[result.predictions["split"].eq("validation")]
sizes = validation.groupby("possession_id").size()
for target in [1, 0]:
    candidates = validation.loc[
        validation["target"].eq(target)
        & validation["possession_id"].map(sizes).ge(3)
    ]
    possession_id = candidates["possession_id"].iloc[0]
    chosen = candidates.loc[candidates["possession_id"].eq(possession_id)]
    shown = chosen.merge(
        result.examples[["event_id", *FEATURE_COLUMNS]], on="event_id", validate="one_to_one"
    )
    print("Recorded subsequent shot:", bool(target))
    display(shown[["event_index", *FEATURE_COLUMNS, "shot_probability", "target"]].round(4))

Recorded subsequent shot: True


,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,shot_probability,target
0,0,mark,0.075,0.3876,0,0,0.3169,1
1,1,kick,0.0375,0.2946,2,2,0.1562,1
2,2,loose_ball_get,0.175,-0.186,5,7,0.2331,1
3,3,handball,0.225,-0.1395,0,7,0.2348,1
4,4,receive,0.325,-0.062,2,9,0.4058,1
5,5,handball,0.3375,-0.0775,0,9,0.2987,1
6,6,receive,0.3625,-0.155,1,10,0.4383,1
7,7,handball,0.3875,-0.124,2,12,0.3312,1
8,8,receive,0.475,-0.0155,2,14,0.5264,1


Recorded subsequent shot: False


,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,shot_probability,target
0,0,loose_ball_get,0.35,-0.0155,0,0,0.3096,0
1,1,handball,0.3375,-0.0155,1,1,0.2599,0
2,2,receive,0.275,0.0,1,2,0.3456,0
3,3,kick,0.2875,0.0155,1,3,0.2641,0


## 10. Model and evaluation artifacts

The export includes the fitted pipeline, current-state examples, match assignments,
predictions, metrics, calibration bins, coefficient table and exclusion audit. The summary
records configuration, versions and source/input hashes. Rerunning this cell
replaces the derived experiment artifacts only.

The raw dataset and earlier notebooks are unchanged. These results establish
a reproducible baseline for later model comparisons; domain review of the
preprocessing rules remains necessary.

In [10]:
output_dir = project_root / "artifacts" / f"logistic-regression-{SEASON}"
save_baseline(result, output_dir, provenance, overwrite=True)
print("Saved experiment:", output_dir.relative_to(project_root).as_posix())
print("Files:")
for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(" ", path.name)
print("Model and evaluation artifacts saved.")

Saved experiment: artifacts/logistic-regression-2023
Files:
  calibration.parquet
  coefficients.parquet
  coverage.parquet
  examples.parquet
  match_split.parquet
  metrics.parquet
  model.pkl
  possession_audit.parquet
  predictions.parquet
  summary.json
Model and evaluation artifacts saved.
